In [2]:
import ast
import math
import pickle
from pathlib import Path
from collections import defaultdict, Counter

from tqdm import tqdm
import bm25s
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## 1. Vizualizarea datelor

In [3]:
df = pd.read_json("data/companies.jsonl", lines=True)

In [4]:
summary = []

for col in df.columns:
    non_null = df[col].dropna()
    type_counts = Counter(non_null.apply(lambda x: type(x).__name__))

    summary.append({
        "column": col,
        "total_rows": len(df),
        "non_null_count": len(non_null),
        "null_count": df[col].isna().sum(),
        "type_counts": dict(type_counts)
    })

summary_df = pd.DataFrame(summary)
summary_df

,column,total_rows,non_null_count,null_count,type_counts
0,website,477,439,38,{'str': 439}
1,operational_name,477,475,2,{'str': 475}
2,year_founded,477,346,131,{'float': 346}
3,address,477,477,0,"{'str': 64, 'dict': 413}"
4,employee_count,477,289,188,{'float': 289}
5,revenue,477,384,93,{'float': 384}
6,primary_naics,477,477,0,"{'str': 64, 'dict': 413}"
7,description,477,477,0,{'str': 477}
8,business_model,477,477,0,{'list': 477}
9,target_markets,477,477,0,{'list': 477}


In [5]:
dict_cols = ["address", "primary_naics", "secondary_naics"]

for col in dict_cols:
    key_type_counts = defaultdict(Counter)

    for x in df[col].dropna():
        if isinstance(x, dict):
            for k, v in x.items():
                key_type_counts[k][type(v).__name__] += 1

    print(f"\n=== {col} ===")
    for k, counter in key_type_counts.items():
        print(f"{k}: {dict(counter)}")


=== address ===
country_code: {'str': 413}
latitude: {'float': 410, 'NoneType': 3}
longitude: {'float': 410, 'NoneType': 3}
region_name: {'str': 402, 'NoneType': 11}
town: {'str': 398, 'NoneType': 15}
country_name: {'str': 20}
county: {'NoneType': 17, 'str': 3}
house_number: {'str': 17, 'NoneType': 3}
postcode: {'str': 20}
raw_text: {'str': 20}
region_code: {'str': 20}
street: {'str': 17, 'NoneType': 3}
suburb: {'NoneType': 12, 'str': 8}

=== primary_naics ===
code: {'str': 413}
label: {'str': 413}
share: {'float': 20}

=== secondary_naics ===
code: {'str': 11}
label: {'str': 11}
share: {'float': 11}


In [6]:
row = df.iloc[0]

print("ADDRESS RAW:", row["address"])
print("ADDRESS TYPE:", type(row["address"]))

print("\nPRIMARY_NAICS RAW:", row["primary_naics"])
print("PRIMARY_NAICS TYPE:", type(row["primary_naics"]))

print("\nSECONDARY_NAICS RAW:", row["secondary_naics"])
print("SECONDARY_NAICS TYPE:", type(row["secondary_naics"]))

ADDRESS RAW: {'country_code': 'ro', 'latitude': 44.4775537, 'longitude': 26.0713416, 'region_name': 'Bucharest', 'town': 'Bucharest'}
ADDRESS TYPE: <class 'str'>

PRIMARY_NAICS RAW: {'code': '324110', 'label': 'Petroleum Refineries'}
PRIMARY_NAICS TYPE: <class 'str'>

SECONDARY_NAICS RAW: None
SECONDARY_NAICS TYPE: <class 'NoneType'>


## 2. Corectarea tipurilor de date

In [7]:
def is_missing(x):
    return x is None or (isinstance(x, float) and math.isnan(x))

def parse_if_literal(x):
    if is_missing(x):
        return x
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                return ast.literal_eval(s)
            except (ValueError, SyntaxError):
                return x
    return x

In [8]:
cols_to_parse = [
    "address",
    "primary_naics",
    "secondary_naics",
    "business_model",
    "target_markets",
    "core_offerings",
]

for col in cols_to_parse:
    df[col] = df[col].apply(parse_if_literal)

In [9]:
summary = []

for col in df.columns:
    non_null = df[col].dropna()
    type_counts = Counter(non_null.apply(lambda x: type(x).__name__))

    summary.append({
        "column": col,
        "total_rows": len(df),
        "non_null_count": len(non_null),
        "null_count": df[col].isna().sum(),
        "type_counts": dict(type_counts)
    })

summary_df = pd.DataFrame(summary)
summary_df

,column,total_rows,non_null_count,null_count,type_counts
0,website,477,439,38,{'str': 439}
1,operational_name,477,475,2,{'str': 475}
2,year_founded,477,346,131,{'float': 346}
3,address,477,477,0,{'dict': 477}
4,employee_count,477,289,188,{'float': 289}
5,revenue,477,384,93,{'float': 384}
6,primary_naics,477,477,0,{'dict': 477}
7,description,477,477,0,{'str': 477}
8,business_model,477,477,0,{'list': 477}
9,target_markets,477,477,0,{'list': 477}


In [10]:
dict_cols = ["address", "primary_naics", "secondary_naics"]

for col in dict_cols:
    key_type_counts = defaultdict(Counter)

    for x in df[col].dropna():
        if isinstance(x, dict):
            for k, v in x.items():
                key_type_counts[k][type(v).__name__] += 1

    print(f"\n=== {col} ===")
    for k, counter in key_type_counts.items():
        print(f"{k}: {dict(counter)}")


=== address ===
country_code: {'str': 477}
latitude: {'float': 474, 'NoneType': 3}
longitude: {'float': 474, 'NoneType': 3}
region_name: {'str': 464, 'NoneType': 13}
town: {'str': 460, 'NoneType': 17}
country_name: {'str': 20}
county: {'NoneType': 17, 'str': 3}
house_number: {'str': 17, 'NoneType': 3}
postcode: {'str': 20}
raw_text: {'str': 20}
region_code: {'str': 20}
street: {'str': 17, 'NoneType': 3}
suburb: {'NoneType': 12, 'str': 8}

=== primary_naics ===
code: {'str': 477}
label: {'str': 477}
share: {'float': 20}

=== secondary_naics ===
code: {'str': 11}
label: {'str': 11}
share: {'float': 11}


In [11]:
row = df.iloc[0]

print("ADDRESS RAW:", row["address"])
print("ADDRESS TYPE:", type(row["address"]))

print("\nPRIMARY_NAICS RAW:", row["primary_naics"])
print("PRIMARY_NAICS TYPE:", type(row["primary_naics"]))

print("\nSECONDARY_NAICS RAW:", row["secondary_naics"])
print("SECONDARY_NAICS TYPE:", type(row["secondary_naics"]))

ADDRESS RAW: {'country_code': 'ro', 'latitude': 44.4775537, 'longitude': 26.0713416, 'region_name': 'Bucharest', 'town': 'Bucharest'}
ADDRESS TYPE: <class 'dict'>

PRIMARY_NAICS RAW: {'code': '324110', 'label': 'Petroleum Refineries'}
PRIMARY_NAICS TYPE: <class 'dict'>

SECONDARY_NAICS RAW: None
SECONDARY_NAICS TYPE: <class 'NoneType'>


## 3. Functii pentru normalizare

In [12]:
def as_clean_str(x):
    if is_missing(x):
        return ""
    return str(x).strip()

def list_field_to_text(values):
    if is_missing(values):
        return ""

    if isinstance(values, list):
        parts = [as_clean_str(v) for v in values]
        parts = [p for p in parts if p]
        return " ".join(parts)

    return as_clean_str(values)

def naics_to_text(value, include_code=True):
    if is_missing(value):
        return ""

    if isinstance(value, str):
        return value.strip()

    if isinstance(value, dict):
        parts = []

        label = value.get("label")
        code = value.get("code")

        if not is_missing(label):
            parts.append(str(label).strip())

        if include_code and not is_missing(code):
            parts.append(str(code).strip())

        return " ".join(parts)

    return as_clean_str(value)

def address_to_text(value):
    if is_missing(value):
        return ""

    if isinstance(value, str):
        return value.strip()

    if isinstance(value, dict):
        if not is_missing(value.get("raw_text")):
            return str(value["raw_text"]).strip()
        
        useful_keys = [
            "street",
            "house_number",
            "suburb",
            "town",
            "county",
            "region_name",
            "country_name",
            "country_code",
            "postcode",
        ]

        parts = []
        for key in useful_keys:
            field_value = value.get(key)
            if not is_missing(field_value):
                parts.append(str(field_value).strip())
        return " ".join(parts)

    return as_clean_str(value)

## 4. Construirea textului companiei

In [13]:
def build_company_text(row):
    parts = [
        as_clean_str(row.get("operational_name")),
        as_clean_str(row.get("website")),
        address_to_text(row.get("address")),
        naics_to_text(row.get("primary_naics")),
        naics_to_text(row.get("secondary_naics")),
        as_clean_str(row.get("description")),
        list_field_to_text(row.get("business_model")),
        list_field_to_text(row.get("target_markets")),
        list_field_to_text(row.get("core_offerings")),
    ]

    parts = [p for p in parts if p]
    return " | ".join(parts)

In [14]:
df["company_text"] = df.apply(build_company_text, axis=1)

output_dir = Path("artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

print("Saving processed dataframe...")
df.to_pickle(output_dir / "companies_processed.pkl")

Saving processed dataframe...


In [15]:
cols = [
    "operational_name",
    "website",
    "address",
    "primary_naics",
    "secondary_naics",
    "description",
    "business_model",
    "target_markets",
    "core_offerings",
    "company_text",
]

print(df[cols].head(5).to_string())

  operational_name        website                                                                                                                           address                                                                           primary_naics secondary_naics                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## 5. Funcțiile principale de pregătire

In [16]:
'''def prepare_index(
    input_path="data/companies.jsonl",
    output_dir="artifacts",
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Loading data...")
    df = pd.read_json(input_path, lines=True)

    print("Building company_text...")
    df["company_text"] = df.apply(build_company_text, axis=1)

    print("Saving processed dataframe...")
    df.to_pickle(output_dir / "companies_processed.pkl")

    print("Building BM25 index...")
    corpus_texts = df["company_text"].fillna("").tolist()
    corpus_tokens = bm25s.tokenize(corpus_texts)

    bm25 = bm25s.BM25()
    bm25.index(corpus_tokens)

    with open(output_dir / "bm25_tokens.pkl", "wb") as f:
        pickle.dump(corpus_tokens, f)

    with open(output_dir / "bm25_index.pkl", "wb") as f:
        pickle.dump(bm25, f)

    print("Loading embedding model...")
    embedder = SentenceTransformer(embedding_model_name)

    print("Encoding company texts...")
    company_embeddings = embedder.encode(
        corpus_texts,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    print("Saving embeddings...")
    np.save(output_dir / "company_embeddings.npy", company_embeddings)

    print("Building FAISS index...")
    dim = company_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(company_embeddings)

    print("Saving FAISS index...")
    faiss.write_index(index, str(output_dir / "company_faiss.index"))

    print("Saving metadata...")
    metadata = {
        "embedding_model_name": embedding_model_name,
        "num_companies": len(df),
        "embedding_dim": dim,
    }
    with open(output_dir / "metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)

    print("Done.")'''

'def prepare_index(\n    input_path="data/companies.jsonl",\n    output_dir="artifacts",\n    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",\n):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    print("Loading data...")\n    df = pd.read_json(input_path, lines=True)\n\n    print("Building company_text...")\n    df["company_text"] = df.apply(build_company_text, axis=1)\n\n    print("Saving processed dataframe...")\n    df.to_pickle(output_dir / "companies_processed.pkl")\n\n    print("Building BM25 index...")\n    corpus_texts = df["company_text"].fillna("").tolist()\n    corpus_tokens = bm25s.tokenize(corpus_texts)\n\n    bm25 = bm25s.BM25()\n    bm25.index(corpus_tokens)\n\n    with open(output_dir / "bm25_tokens.pkl", "wb") as f:\n        pickle.dump(corpus_tokens, f)\n\n    with open(output_dir / "bm25_index.pkl", "wb") as f:\n        pickle.dump(bm25, f)\n\n    print("Loading embedding model...")\n    embedder = SentenceTra

In [17]:
def prepare_bm25():
    print("Building BM25 index...")
    corpus_texts = df["company_text"].fillna("").tolist()
    corpus_tokens = bm25s.tokenize(corpus_texts)

    bm25 = bm25s.BM25()
    bm25.index(corpus_tokens)

    with open(output_dir / "bm25_tokens.pkl", "wb") as f:
        pickle.dump(corpus_tokens, f)

    with open(output_dir / "bm25_index.pkl", "wb") as f:
        pickle.dump(bm25, f)
    
    print("Done.")

In [18]:
def prepare_embeddings( embedding_model_name="sentence-transformers/all-MiniLM-L6-v2" ):
    corpus_texts = df["company_text"].fillna("").tolist()

    print("Loading embedding model...")
    embedder = SentenceTransformer(embedding_model_name)

    print("Encoding company texts...")
    company_embeddings = embedder.encode(
        corpus_texts,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    print("Saving embeddings...")
    np.save(output_dir / "company_embeddings.npy", company_embeddings)

    print("Building FAISS index...")
    dim = company_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(company_embeddings)

    print("Saving FAISS index...")
    faiss.write_index(index, str(output_dir / "company_faiss.index"))

    print("Saving metadata...")
    metadata = {
        "embedding_model_name": embedding_model_name,
        "num_companies": len(df),
        "embedding_dim": dim,
    }
    with open(output_dir / "metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)

    print("Done.")

In [19]:
prepare_bm25()
prepare_embeddings()

Building BM25 index...


Done.
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9914.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding company texts...


Batches: 100%|██████████| 15/15 [00:01<00:00,  9.49it/s]

Saving embeddings...
Building FAISS index...
Saving FAISS index...
Saving metadata...
Done.


In [20]:
df_proc = pd.read_pickle("artifacts/companies_processed.pkl")
print(df_proc.shape)
print(df_proc.columns)

(477, 14)
Index(['website', 'operational_name', 'year_founded', 'address',
       'employee_count', 'revenue', 'primary_naics', 'description',
       'business_model', 'target_markets', 'core_offerings', 'is_public',
       'secondary_naics', 'company_text'],
      dtype='str')


In [21]:
df_proc[["operational_name", "address", "description", "company_text"]].head(10)

,operational_name,address,description,company_text
0,Rompetrol,"{'country_code': 'ro', 'latitude': 44.4775537,...",Rompetrol is a Romanian company specialized in...,Rompetrol | rompetrol.ro | Bucharest Bucharest...
1,Unilever,"{'country_code': 'ro', 'latitude': 44.4761769,...","Unilever South Central Europe SA, DBA Unilever...",Unilever | unilever.ro | Bucharest Bucharest r...
2,Rompetrol,"{'country_code': 'ro', 'latitude': 44.47775356...",Rompetrol is a Romanian energy company engaged...,Rompetrol | rompetrol.com | Bucharest Buchares...
3,CBRE Romania,"{'country_code': 'ro', 'latitude': 44.4540414,...",CBRE Romania is a Romanian company engaged in ...,CBRE Romania | cbre.ro | Bucharest Bucharest r...
4,TIMAC AGRO,"{'country_code': 'ro', 'latitude': 44.4792186,...",TIMAC AGRO S.R.L. is a French agricultural com...,TIMAC AGRO | timacagro.com | Bucharest Buchare...
5,OMV,"{'country_code': 'ro', 'latitude': 44.490331, ...",OMV PETROM Marketing SRL is a Romanian company...,OMV | omv.ro | Bucharest Bucharest ro | Other ...
6,Compania Națională de Căi Ferate CFR,"{'country_code': 'ro', 'latitude': 44.4493708,...","Compania Națională de Căi Ferate ”CFR” – SA, k...",Compania Națională de Căi Ferate CFR | cfr.ro ...
7,Mercedes-Benz Trucks & Buses Romania,"{'country_code': 'ro', 'latitude': 44.49860626...","Daimler Truck & Bus Romania S.R.L., DBA Merced...",Mercedes-Benz Trucks & Buses Romania | mercede...
8,Dacia,"{'country_code': 'ro', 'latitude': 44.429628, ...","Renault Commercial Roumanie S.R.L., DBA Dacia,...",Dacia | dacia.ro | Bucharest Bucharest ro | Au...
9,Fildas Trading,"{'country_code': 'ro', 'latitude': 44.4781612,...",Fildas Trading SRL is a Romanian company engag...,Fildas Trading | fildas.ro | Bucharest Buchare...


In [22]:
with open("artifacts/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print(metadata)

{'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'num_companies': 477, 'embedding_dim': 384}


In [23]:
embeddings = np.load("artifacts/company_embeddings.npy")
print(embeddings.shape)
print(embeddings[0][:10])   # primele 10 valori din primul embedding

(477, 384)
[-0.10277884 -0.00499695 -0.00820839  0.05740421 -0.04844     0.00664749
 -0.05767528  0.08829867 -0.08050168 -0.19097486]


In [24]:
index = faiss.read_index("artifacts/company_faiss.index")
print("Num vectors in index:", index.ntotal)

Num vectors in index: 477


In [ ]:
with open("artifacts/bm25_tokens.pkl", "rb") as f:
    corpus_tokens = pickle.load(f)

print(type(corpus_tokens))
print(len(corpus_tokens))
print(corpus_tokens[0][:30])       # primii tokeni din prima companie

<class 'bm25s.tokenization.Tokenized'>
2
[[0, 0, 1, 2, 2, 1, 3, 4, 5, 0, 6, 7, 8, 3, 9, 10, 11, 12, 13, 14, 7, 15, 16, 17, 18, 19, 20, 4, 21, 22, 23, 24, 25, 26, 27, 28, 29, 0, 30, 31, 27, 14, 32, 13, 33, 34, 35, 36, 37, 38, 39, 40, 41, 32, 42, 43, 43, 44, 45, 46, 43, 47, 48, 28, 27, 49, 13, 50, 12, 37, 39, 51, 35, 10, 50, 42, 34, 52, 53, 3, 9, 12, 54, 13, 44, 55, 56, 57, 12, 58, 28, 52, 59, 40, 60, 61, 62, 63, 28, 64, 35, 27, 13, 33, 35, 32, 13, 10, 50, 65, 66, 67, 56, 35], [68, 68, 1, 2, 2, 1, 69, 70, 71, 14, 72, 73, 74, 68, 75, 76, 77, 78, 79, 68, 6, 7, 8, 12, 44, 47, 80, 81, 82, 83, 84, 14, 7, 85, 6, 86, 87, 68, 88, 89, 15, 90, 68, 91, 30, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 44, 43, 43, 43, 47, 32, 47, 80, 47, 80, 12, 44, 103, 104, 105, 106, 35, 93, 107, 108, 35, 93, 109, 108, 35, 84, 14, 12, 44, 47, 80, 110, 59, 58, 50, 59, 82, 83, 14, 12, 44], [0, 0, 111, 2, 2, 1, 3, 4, 5, 0, 6, 28, 7, 112, 9, 113, 44, 11, 114, 115, 114, 35, 66, 116, 7, 117, 118, 119, 14, 120, 121, 43,

In [26]:
with open("artifacts/bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)

print(type(bm25))

<class 'bm25s.BM25'>
